In [2]:
import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from liffile import LifFile
import napari
import numpy as np
import pandas as pd
from liffile import LifFile
from magicgui import magicgui
from magicgui.widgets import Container, TextEdit
from matplotlib.colors import to_rgba
from skimage import util
from skimage.measure import marching_cubes, mesh_surface_area, label, regionprops_table
from skimage.transform import rescale
from napari.utils.colormaps import DirectLabelColormap
from IPython.display import display
import imagej
from imagej import Mode

import scyjava as sj

In [3]:
class PerfusionNapariGUIApp:
    def __init__(self):
        self.viewer = napari.Viewer()

        self.ij = None
        self.IJ = None
        self.Duplicator = None
        self.WekaSegmentation = None
        self.ImagePlus = None
        self.ImageStack = None
        self.FloatProcessor = None

        self._selected_lif_path: Optional[Path] = None
        self._classifier_path: Optional[Path] = None
        self._dextran_channels: List[int] = [0]
        self._dextran_channel_names: Dict[int, str] = {0: "Dextran 1"}
        self._permeability_divisor_s: float = 360.0
        self._image_choice_map: Dict[str, int] = {}
        self._results: List[pd.DataFrame] = []
        self._results_df: Optional[pd.DataFrame] = None
        self._last_channel_info_signature: Optional[Tuple[Tuple[int, ...], int]] = None

        self.log_output = TextEdit(value="")
        self.log_output.min_height = 140
        self.log_output.max_height = 320
        try:
            self.log_output.native.setReadOnly(True)
        except Exception:
            pass

        self.load_images = magicgui(
            self._list_images,
            lif_path={"label": "Select .lif", "mode": "r", "filter": "*.lif"},
            classifier_path={"label": "Weka classifier", "mode": "r"},
            call_button="Load images",
        )

        self.channel_config = magicgui(
            self._channel_config_placeholder,
            dextran_channel_1={"label": "Dextran channel 1", "widget_type": "ComboBox", "choices": ["0"], "value": "0"},
            dextran_name_1={"label": "Name channel 1", "widget_type": "LineEdit", "value": "Dextran 1"},
            dextran_channel_2={"label": "Dextran channel 2 (optional)", "widget_type": "ComboBox", "choices": ["None", "0"], "value": "None"},
            dextran_name_2={"label": "Name channel 2", "widget_type": "LineEdit", "value": ""},
            dextran_channel_3={"label": "Dextran channel 3 (optional)", "widget_type": "ComboBox", "choices": ["None", "0"], "value": "None"},
            dextran_name_3={"label": "Name channel 3", "widget_type": "LineEdit", "value": ""},
            permeability_divisor_s={"label": "Time separation initial→final (s)", "widget_type": "LineEdit", "value": "360"},
            call_button=False,
        )

        self.run_single = magicgui(
            self._run_single_image,
            image_choice={"label": "Image", "choices": ["(load images)"], "widget_type": "ComboBox"},
            call_button="Analyse Single Image",
        )

        self.run_all = magicgui(
            self._run_all_images,
            output_csv={"label": "Output CSV", "mode": "w", "value": "perfusion_all_images.csv"},
            call_button="Analyse All Images",
        )

        self.save_results = magicgui(
            self._save_results,
            output_csv={"label": "Save current results", "mode": "w", "value": "perfusion_results.csv"},
            call_button="Save This Result",
        )

        self.load_images.lif_path.changed.connect(self._update_dextran_channel_choices)
        self._update_dextran_channel_choices()

        log_panel = Container(widgets=[self.log_output])
        self.viewer.window.add_dock_widget(self.load_images, area="right")
        self.viewer.window.add_dock_widget(self.channel_config, area="right")
        self.viewer.window.add_dock_widget(self.run_single, area="right")
        self.viewer.window.add_dock_widget(self.run_all, area="right")
        self.viewer.window.add_dock_widget(self.save_results, area="right")
        self.viewer.window.add_dock_widget(log_panel, area="right")

    @staticmethod
    def _channel_config_placeholder(
        dextran_channel_1="0",
        dextran_name_1="Dextran 1",
        dextran_channel_2="None",
        dextran_name_2="",
        dextran_channel_3="None",
        dextran_name_3="",
        permeability_divisor_s="360",
    ):
        _ = (
            dextran_channel_1,
            dextran_name_1,
            dextran_channel_2,
            dextran_name_2,
            dextran_channel_3,
            dextran_name_3,
            permeability_divisor_s,
        )
        return None

    def _init_imagej_bindings(self, mode: Mode = Mode.HEADLESS, fiji_path: Optional[str] = None):
        if self.ij is not None:
            return

        target = "sc.fiji:fiji"
        if fiji_path:
            local_fiji = Path(str(fiji_path).strip())
            if not local_fiji.exists():
                raise FileNotFoundError(f"Fiji path does not exist: {local_fiji}")
            target = str(local_fiji)

        self.ij = imagej.init(target, mode=mode, add_legacy=True)
        self.IJ = sj.jimport("ij.IJ")
        self.Duplicator = sj.jimport("ij.plugin.Duplicator")
        self.WekaSegmentation = sj.jimport("trainableSegmentation.WekaSegmentation")
        self.ImagePlus = sj.jimport("ij.ImagePlus")
        self.ImageStack = sj.jimport("ij.ImageStack")
        self.FloatProcessor = sj.jimport("ij.process.FloatProcessor")

    def _append_log(self, message: str):
        self.log_output.value = (
            self.log_output.value.rstrip() + "\n" + message if self.log_output.value else message
        )

    @staticmethod
    def _normalize_csv_path(path_value: Path) -> Path:
        path = Path(path_value)
        if path.suffix.lower() != ".csv":
            return path.with_suffix(".csv")
        return path

    @staticmethod
    def _parse_optional_channel(value) -> Optional[int]:
        if value is None:
            return None
        text = str(value).strip()
        if text == "" or text.lower() == "none":
            return None
        return int(text)

    @staticmethod
    def _safe_channel_name(name_value: str, order_index: int) -> str:
        text = str(name_value).strip()
        if text:
            return text
        return f"Dextran {order_index}"

    @staticmethod
    def _widget_text(widget) -> str:
        try:
            native = getattr(widget, "native", None)
            if native is not None:
                if hasattr(native, "currentText"):
                    return str(native.currentText()).strip()
                if hasattr(native, "text"):
                    return str(native.text()).strip()
        except Exception:
            pass
        try:
            return str(widget.value).strip()
        except Exception:
            return ""

    def _read_permeability_divisor_from_widget(self) -> float:
        raw_value = self._widget_text(self.channel_config.permeability_divisor_s)
        try:
            divisor = float(raw_value)
            if divisor <= 0:
                raise ValueError("must be > 0")
            return divisor
        except Exception:
            self._append_log(f"[WARN] Invalid time separation '{raw_value}'. Using default 360 s.")
            try:
                self.channel_config.permeability_divisor_s.value = "360"
            except Exception:
                pass
            return 360.0

    def _read_channel_config_from_widgets(self) -> Tuple[List[int], Dict[int, str]]:
        ch1_raw = self._widget_text(self.channel_config.dextran_channel_1)
        name1_raw = self._widget_text(self.channel_config.dextran_name_1)
        ch2_raw = self._widget_text(self.channel_config.dextran_channel_2)
        name2_raw = self._widget_text(self.channel_config.dextran_name_2)
        ch3_raw = self._widget_text(self.channel_config.dextran_channel_3)
        name3_raw = self._widget_text(self.channel_config.dextran_name_3)

        channel_name_inputs = [
            (
                int(str(ch1_raw)),
                self._safe_channel_name(name1_raw, 1),
            ),
        ]

        opt2 = self._parse_optional_channel(ch2_raw)
        if opt2 is not None:
            channel_name_inputs.append(
                (opt2, self._safe_channel_name(name2_raw, 2))
            )

        opt3 = self._parse_optional_channel(ch3_raw)
        if opt3 is not None:
            channel_name_inputs.append(
                (opt3, self._safe_channel_name(name3_raw, 3))
            )

        channels: List[int] = []
        channel_name_map: Dict[int, str] = {}
        for channel_index, channel_name in channel_name_inputs:
            if channel_index >= 0 and channel_index not in channels:
                channels.append(channel_index)
                channel_name_map[channel_index] = channel_name

        if not channels:
            channels = [0]
            channel_name_map = {0: "Dextran 1"}

        return channels, channel_name_map

    def _update_dextran_channel_choices(self, lif_path_value=None, *_):
        if not lif_path_value:
            lif_path_value = self.load_images.lif_path.value
        if not lif_path_value:
            return

        lif_path = Path(str(lif_path_value))
        if not lif_path.exists():
            return

        try:
            with LifFile(lif_path) as lif:
                channel_counts = [self._channel_count_from_image(img) for img in lif.images]
                if not channel_counts:
                    return
                max_channels = max(1, int(max(channel_counts)))
                if len(set(channel_counts)) > 1:
                    signature = (tuple(sorted(set(channel_counts))), max_channels)
                    if self._last_channel_info_signature != signature:
                        self._append_log(
                            f"[INFO] Channel counts vary across images {sorted(set(channel_counts))}; allowing GUI choices 0..{max_channels-1}. Channels unavailable in some images will be skipped."
                        )
                        self._last_channel_info_signature = signature
        except Exception:
            return

        valid_choices = [str(i) for i in range(max_channels)]

        self.channel_config.dextran_channel_1.choices = valid_choices
        if str(self.channel_config.dextran_channel_1.value) not in valid_choices:
            self.channel_config.dextran_channel_1.value = valid_choices[0]

        optional_choices = ["None"] + valid_choices

        old2 = str(self.channel_config.dextran_channel_2.value)
        self.channel_config.dextran_channel_2.choices = optional_choices
        self.channel_config.dextran_channel_2.value = old2 if old2 in optional_choices else "None"

        old3 = str(self.channel_config.dextran_channel_3.value)
        self.channel_config.dextran_channel_3.choices = optional_choices
        self.channel_config.dextran_channel_3.value = old3 if old3 in optional_choices else "None"

    @staticmethod
    def _step(axis, xa):
        if axis not in xa.coords or xa.coords[axis].size < 2:
            return None
        return float(xa.coords[axis][1] - xa.coords[axis][0])

    @staticmethod
    def _convert(img, target_type_min, target_type_max, target_type):
        imin = float(img.min())
        imax = float(img.max())
        if imax == imin:
            return np.full_like(img, fill_value=target_type_min, dtype=target_type)
        a = (target_type_max - target_type_min) / (imax - imin)
        b = target_type_max - a * imax
        return (a * img + b).astype(target_type)

    def _numpy_to_imageplus(self, arr: np.ndarray, title="image"):
        if arr.ndim == 2:
            y, x = arr.shape
            pix = np.asarray(arr, dtype=np.float32).ravel()
            java_floats = sj.jarray("f", pix.size)
            for i, v in enumerate(pix.tolist()):
                java_floats[i] = float(v)
            fp = self.FloatProcessor(x, y, java_floats)
            return self.ImagePlus(title, fp)

        if arr.ndim == 3:
            z, y, x = arr.shape
            stack = self.ImageStack(x, y)
            for zi in range(z):
                pix = np.asarray(arr[zi], dtype=np.float32).ravel()
                java_floats = sj.jarray("f", pix.size)
                for i, v in enumerate(pix.tolist()):
                    java_floats[i] = float(v)
                fp = self.FloatProcessor(x, y, java_floats)
                stack.addSlice(fp)
            return self.ImagePlus(title, stack)

        raise ValueError(f"Unsupported shape: {arr.shape}")

    @staticmethod
    def _imageplus_to_numpy(imp) -> np.ndarray:
        w, h = imp.getWidth(), imp.getHeight()
        n = imp.getStackSize()
        stack = imp.getStack()
        out = []
        for z in range(1, n + 1):
            ip = stack.getProcessor(z)
            pix = np.array(ip.getPixels()).reshape(h, w)
            out.append(pix)
        return np.stack(out, axis=0) if n > 1 else out[0]

    @staticmethod
    def _extract_tzyx_for_channel(image: np.ndarray, dims, channel: int) -> np.ndarray:
        dims = tuple(dims)
        if dims == ("T", "Z", "Y", "X"):
            if channel != 0:
                raise ValueError("Single-channel image; only channel 0 is valid")
            return image
        if dims == ("T", "C", "Z", "Y", "X"):
            return image[:, channel, :, :, :]
        raise ValueError(f"Unsupported image dims: {dims}")

    @staticmethod
    def _channel_count_from_image(img) -> int:
        dims = tuple(getattr(img, "dims", ()))
        shape = tuple(getattr(img, "shape", ()))
        if dims == ("T", "C", "Z", "Y", "X") and len(shape) == 5:
            return int(shape[1])
        if dims == ("T", "Z", "Y", "X") and len(shape) == 4:
            return 1
        return 1

    def _apply_weka_with_exact_preprocessing(self, image: np.ndarray, model_path: str) -> np.ndarray:
        if self.ij is None:
            self._append_log("[INFO] Initializing ImageJ in HEADLESS mode for analysis...")
            self._init_imagej_bindings(mode=Mode.HEADLESS)

        imp = self._numpy_to_imageplus(image, title="image")
        dup = self.Duplicator().run(imp)
        self.IJ.run(dup, "8-bit", "")
        self.IJ.run(dup, "Auto Threshold", "method=Otsu stack")
        self.IJ.run(dup, "Erode (3D)", "iso=255")

        try:
            seg = self.WekaSegmentation(dup)
            seg.loadClassifier(model_path)
            out_imp = seg.applyClassifier(dup, 0, False)
            if out_imp is None:
                out_imp = seg.getClassifiedImage()
            if out_imp is None:
                raise RuntimeError("No classified image returned (out_imp is None).")
            return self._imageplus_to_numpy(out_imp)
        finally:
            try:
                self.IJ.run("Close All")
            except Exception:
                pass

    def _segment_clean_mask(self, t0: np.ndarray) -> np.ndarray:
        gel_matrix = self._apply_weka_with_exact_preprocessing(t0, str(self._classifier_path))
        vasculature_segmentation = (gel_matrix == 0).astype(int)
        vasculature_labels = label(vasculature_segmentation)
        table = regionprops_table(vasculature_labels, properties=("label", "area"))

        condition = table["area"] >= 20
        input_labels = table["label"]
        output_labels = input_labels * condition
        output_labels = util.map_array(vasculature_labels, input_labels, output_labels)
        return output_labels > 0

    def _quantify_permeability(
        self,
        channel_tzyx: np.ndarray,
        clean_mask: np.ndarray,
        x_um: float,
        z_um: float,
        lif_name: str,
        image_name: str,
        channel_index: int,
        channel_name: str,
        permeability_divisor_s: float,
    ) -> pd.DataFrame:
        t0 = self._convert(channel_tzyx[0, :, :, :], 0, 255, np.uint8)
        t_final = self._convert(channel_tzyx[-1, :, :, :], 0, 255, np.uint8)

        final_vascular_intensity = np.float64(np.sum(t_final[clean_mask == 1]))
        final_gel_intensity = np.float64(np.sum(t_final[clean_mask == 0]))
        initial_vascular_intensity = np.float64(np.sum(t0[clean_mask == 1]))
        initial_gel_intensity = np.float64(np.sum(t0[clean_mask == 0]))

        vascular_intensities: List[np.float64] = []
        for t_index in range(channel_tzyx.shape[0]):
            t_img = self._convert(channel_tzyx[t_index, :, :, :], 0, 255, np.uint8)
            vascular_intensities.append(np.float64(np.sum(t_img[clean_mask == 1])))

        intensity_columns = {
            f"intensity(t{t_index + 1})": [vascular_intensities[t_index]]
            for t_index in range(len(vascular_intensities))
        }

        vasculature_increase_flags: List[bool] = []
        vasculature_increase_details: List[str] = []
        for t_index in range(1, len(vascular_intensities)):
            previous_intensity = vascular_intensities[t_index - 1]
            current_intensity = vascular_intensities[t_index]
            if previous_intensity == 0:
                increase_fraction = np.inf if current_intensity > 0 else 0.0
            else:
                increase_fraction = (current_intensity - previous_intensity) / previous_intensity

            if increase_fraction > 0.10:
                vasculature_increase_flags.append(True)
                if np.isfinite(increase_fraction):
                    vasculature_increase_details.append(
                        f"t{t_index}->t{t_index + 1}: +{increase_fraction * 100:.2f}%"
                    )
                else:
                    vasculature_increase_details.append(
                        f"t{t_index}->t{t_index + 1}: +inf% (previous intensity was 0)"
                    )

        has_large_vasculature_increase = bool(any(vasculature_increase_flags))
        large_increase_intervals = "; ".join(vasculature_increase_details)

        rescaled_t0 = rescale(scale=(z_um / x_um, 1, 1), image=t0, anti_aliasing=False)
        rescaled_clean_mask = rescale(
            scale=(z_um / x_um, 1, 1),
            image=clean_mask,
            anti_aliasing=False,
            order=0,
            preserve_range=True,
        ).astype(clean_mask.dtype)

        verts, faces, _, _ = marching_cubes(
            rescaled_clean_mask.astype(np.uint8), level=0.5, spacing=(x_um,) * 3
        )
        vasculature_surface_area = mesh_surface_area(verts, faces)
        vascular_volume = np.sum(rescaled_clean_mask == 1) * (x_um**3)
        total_volume = (
            rescaled_t0.shape[0] * rescaled_t0.shape[1] * rescaled_t0.shape[2] * (x_um**3)
        )
        vasculature_volume_fraction = vascular_volume / total_volume if total_volume > 0 else np.nan
        vasculature_volume_gt_80pct = bool(vasculature_volume_fraction > 0.80) if np.isfinite(vasculature_volume_fraction) else False
        flag_text = (
            f"WARN: Segmented vasculature volume is {vasculature_volume_fraction * 100:.2f}% (>80%)."
            if vasculature_volume_gt_80pct
            else ""
        )
        bleaching_coefficient = initial_vascular_intensity / final_vascular_intensity
        gel_volume = total_volume - vascular_volume
        p = (1 / permeability_divisor_s) * (gel_volume / vasculature_surface_area) * (
            ((bleaching_coefficient * final_gel_intensity) - initial_gel_intensity)
            / (initial_vascular_intensity - initial_gel_intensity)
        )

        return pd.DataFrame(
            {
                "lif_name": [lif_name],
                "image_name": [image_name],
                "dextran_channel": [channel_name],
                "dextran_channel_index": [int(channel_index)],
                "image_shape": [channel_tzyx.shape],
                "num_timepoints": [int(channel_tzyx.shape[0])],
                "final_gel_intensity": [final_gel_intensity],
                "final_vascular_intensity": [final_vascular_intensity],
                "initial_gel_intensity": [initial_gel_intensity],
                "initial_vascular_intensity": [initial_vascular_intensity],
                "vascular_volume_um3": [vascular_volume],
                "vasculature_volume_fraction": [vasculature_volume_fraction],
                "vasculature_volume_gt_80pct": [vasculature_volume_gt_80pct],
                "gel_volume_um3": [gel_volume],
                "vasculature_surface_area_um2": [vasculature_surface_area],
                "bleaching_coefficient": [bleaching_coefficient],
                "time_separation_s": [permeability_divisor_s],
                "p_um/s": [p],
                "p_cm/s": [p * 0.0001],
                "vasculature_increase_gt_10pct_between_timepoints": [has_large_vasculature_increase],
                "vasculature_increase_gt_10pct_intervals": [large_increase_intervals],
                "flag": [flag_text],
                **intensity_columns,
            }
        )

    def _analyze_image(self, lif, image_index: int) -> Tuple[pd.DataFrame, Optional[dict]]:
        img = lif.images[image_index]
        lif_name = "".join(os.path.basename(self._selected_lif_path).lower().replace(".lif", ""))
        image_name = "".join(img.path)

        try:
            image = img.asarray()
            dims = tuple(img.dims)
            channel_count = self._channel_count_from_image(img)

            valid_channels = [ch for ch in self._dextran_channels if ch < channel_count]
            invalid_channels = [ch for ch in self._dextran_channels if ch >= channel_count]
            rows: List[pd.DataFrame] = []

            if invalid_channels:
                pass

            if not valid_channels:
                return pd.DataFrame(), None

            seg_channel = int(valid_channels[0])
            seg_tzyx = self._extract_tzyx_for_channel(image, dims, seg_channel)
            if seg_tzyx.shape[0] < 2:
                return pd.DataFrame(
                    {
                        "lif_name": [lif_name],
                        "image_name": [image_name],
                        "dextran_channel": [self._dextran_channel_names.get(seg_channel, f"Dextran {seg_channel}")],
                        "num_timepoints": [int(seg_tzyx.shape[0])],
                        "flag": ["Need at least 2 timepoints"],
                    }
                ), None

            xa = img.asxarray()
            x_um = self._step("X", xa) * 1e6 if self._step("X", xa) is not None else None
            z_um = self._step("Z", xa) * 1e6 if self._step("Z", xa) is not None else None

            t0_seg = seg_tzyx[0, :, :, :]
            clean_mask = self._segment_clean_mask(t0_seg)

            preview_channels: Dict[int, Dict[str, np.ndarray]] = {}

            for channel_index in valid_channels:
                channel_name = self._dextran_channel_names.get(int(channel_index), f"Dextran {channel_index}")
                try:
                    channel_tzyx = self._extract_tzyx_for_channel(image, dims, int(channel_index))
                    if channel_tzyx.shape[0] < 2:
                        rows.append(
                            pd.DataFrame(
                                {
                                    "lif_name": [lif_name],
                                    "image_name": [image_name],
                                    "dextran_channel": [channel_name],
                                    "dextran_channel_index": [int(channel_index)],
                                    "num_timepoints": [int(channel_tzyx.shape[0])],
                                    "flag": ["Need at least 2 timepoints"],
                                }
                            )
                        )
                        continue

                    preview_channels[int(channel_index)] = {
                        "name": channel_name,
                        "t0": channel_tzyx[0, :, :, :],
                        "tfinal": channel_tzyx[-1, :, :, :],
                    }

                    if x_um is None or z_um is None:
                        rows.append(
                            pd.DataFrame(
                                {
                                    "lif_name": [lif_name],
                                    "image_name": [image_name],
                                    "dextran_channel": [channel_name],
                                    "dextran_channel_index": [int(channel_index)],
                                    "num_timepoints": [int(channel_tzyx.shape[0])],
                                    "flag": ["Missing voxel metadata"],
                                }
                            )
                        )
                        continue

                    rows.append(
                        self._quantify_permeability(
                            channel_tzyx,
                            clean_mask,
                            x_um,
                            z_um,
                            lif_name,
                            image_name,
                            int(channel_index),
                            channel_name,
                            self._permeability_divisor_s,
                        )
                    )
                except Exception as channel_error:
                    rows.append(
                        pd.DataFrame(
                            {
                                "lif_name": [lif_name],
                                "image_name": [image_name],
                                "dextran_channel": [channel_name],
                                "dextran_channel_index": [int(channel_index)],
                                "num_timepoints": [int(channel_tzyx.shape[0])],
                                "flag": [f"FAILED: {type(channel_error).__name__}: {channel_error}"],
                            }
                        )
                    )

            combined = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
            return combined, {"channel_images": preview_channels, "mask": clean_mask}

        except Exception as e:
            return pd.DataFrame(
                {
                    "lif_name": [lif_name],
                    "image_name": [image_name],
                    "dextran_channel": [np.nan],
                    "num_timepoints": [np.nan],
                    "flag": [f"FAILED: {type(e).__name__}: {e}"],
                }
            ), None
        finally:
            try:
                self.IJ.run("Close All")
            except Exception:
                pass

    def _show_single_result_layers(
        self,
        channel_images: Dict[int, Dict[str, np.ndarray]],
        clean_vasculature_segmentation: np.ndarray,
    ):
        remove_names = ["Segmented Vasculature"]
        remove_names.extend([layer.name for layer in self.viewer.layers if layer.name.startswith("Dextran ")])

        for layer_name in remove_names:
            if layer_name in self.viewer.layers:
                self.viewer.layers.remove(self.viewer.layers[layer_name])

        for channel_index in sorted(channel_images.keys()):
            channel_name = channel_images[channel_index]["name"]
            self.viewer.add_image(channel_images[channel_index]["t0"], name=f"Dextran {channel_name} t0")
            self.viewer.add_image(channel_images[channel_index]["tfinal"], name=f"Dextran {channel_name} tfinal")

        self.viewer.add_labels(
            clean_vasculature_segmentation,
            name="Segmented Vasculature",
            colormap=DirectLabelColormap(color_dict={0: (0.0, 0.0, 0.0, 0.0), 1: (1.0, 0.0, 0.0, 1.0)}),
        )

    def _require_loaded_inputs(self) -> bool:
        if self._selected_lif_path is None or not self._selected_lif_path.exists():
            self._append_log("[WARN] Select a .lif and click Load images first.")
            return False
        if self._classifier_path is None or not self._classifier_path.exists():
            self._append_log("[WARN] Select a valid Weka classifier and click Load images.")
            return False
        return True

    def _list_images(self, lif_path: Path = Path(), classifier_path: Path = Path()):
        if not lif_path or not Path(lif_path).exists():
            self._append_log("[WARN] Select a valid .lif file.")
            return None
        if not classifier_path or not Path(classifier_path).exists():
            self._append_log("[WARN] Select a valid Weka classifier.")
            return None

        self._update_dextran_channel_choices(lif_path)
        channels, channel_name_map = self._read_channel_config_from_widgets()

        self._selected_lif_path = Path(lif_path)
        self._classifier_path = Path(classifier_path)
        self._dextran_channels = channels
        self._dextran_channel_names = channel_name_map
        self._permeability_divisor_s = self._read_permeability_divisor_from_widget()

        labels: List[str] = []
        choice_map: Dict[str, int] = {}

        try:
            with LifFile(self._selected_lif_path) as lif:
                for i, img in enumerate(lif.images):
                    name = "".join(getattr(img, "path", ())) or f"Image {i}"
                    label = f"{i}: {name}"
                    labels.append(label)
                    choice_map[label] = i
        except Exception as e:
            self._append_log(f"[ERROR] Failed reading lif: {type(e).__name__}: {e}")
            return None

        if not labels:
            self._image_choice_map = {}
            self.run_single.image_choice.choices = ["(no images)"]
            self.run_single.image_choice.value = "(no images)"
            self._append_log("[WARN] No images found in this .lif.")
            return None

        self._image_choice_map = choice_map
        self.run_single.image_choice.choices = labels
        self.run_single.image_choice.value = labels[0]

        pretty_channels = [f"{idx}:{self._dextran_channel_names.get(idx, f'Dextran {idx}')}" for idx in self._dextran_channels]
        self._append_log(f"[OK] Loaded .lif with {len(labels)} image(s).")
        self._append_log(f"[OK] Using classifier: {self._classifier_path.name}")
        self._append_log(f"[OK] Dextran channels selected: {pretty_channels}")
        self._append_log(f"[OK] Time separation initial→final (s): {self._permeability_divisor_s}")
        self._append_log(
            f"[INFO] Segmentation channel: {self._dextran_channels[0]} ({self._dextran_channel_names.get(self._dextran_channels[0], '')})"
        )
        return labels

    def _run_single_image(self, image_choice: str = "(load images)"):
        if not self._require_loaded_inputs():
            return None

        self._dextran_channels, self._dextran_channel_names = self._read_channel_config_from_widgets()
        self._permeability_divisor_s = self._read_permeability_divisor_from_widget()
        if image_choice not in self._image_choice_map:
            self._append_log("[WARN] Select an image from dropdown.")
            return None

        image_index = self._image_choice_map[image_choice]
        self._append_log(f"[PROGRESS] Running single image_index={image_index}...")
        self._append_log(f"[INFO] Single-image time separation initial→final (s): {self._permeability_divisor_s}")

        with LifFile(self._selected_lif_path) as lif:
            row, preview = self._analyze_image(lif, image_index)

        if preview is not None:
            try:
                self._show_single_result_layers(preview["channel_images"], preview["mask"])
                self._append_log("[OK] Added t0/tfinal dextran layers and Segmented Vasculature to napari viewer.")
            except Exception as e:
                self._append_log(f"[WARN] Could not render preview layers: {type(e).__name__}: {e}")

        self._results.append(row)
        self._results_df = pd.concat(self._results, ignore_index=True)
        self._append_log("[OK] Single image complete.")
        display(self._results_df)
        return row

    def _run_all_images(self, output_csv: Path = Path("perfusion_all_images.csv")):
        if not self._require_loaded_inputs():
            return None

        self._dextran_channels, self._dextran_channel_names = self._read_channel_config_from_widgets()
        selected_channels_pretty = [
            f"{idx}:{self._dextran_channel_names.get(idx, f'Dextran {idx}')}"
            for idx in self._dextran_channels
        ]
        self._append_log(f"[INFO] Batch using dextran channels: {selected_channels_pretty}")
        self._permeability_divisor_s = self._read_permeability_divisor_from_widget()
        self._append_log(f"[INFO] Batch time separation initial→final (s): {self._permeability_divisor_s}")

        output_csv = self._normalize_csv_path(output_csv)
        output_csv.parent.mkdir(parents=True, exist_ok=True)

        rows: List[pd.DataFrame] = []
        with LifFile(self._selected_lif_path) as lif:
            n_images = len(lif.images)
            self._append_log(f"[INFO] Running all images in {self._selected_lif_path.name} (n={n_images})")
            for i in range(n_images):
                if i == 0 or (i + 1) % 10 == 0 or (i + 1) == n_images:
                    self._append_log(f"[PROGRESS] ({i + 1}/{n_images}) image_index={i}")
                row, _ = self._analyze_image(lif, i)
                if row is not None and not row.empty:
                    rows.append(row)

        if not rows:
            self._append_log("[WARN] No results generated.")
            return None

        batch_df = pd.concat(rows, ignore_index=True)
        batch_df.to_csv(output_csv, index=False)
        try:
            if "dextran_channel" in batch_df.columns and "p_um/s" in batch_df.columns:
                counts = (
                    batch_df.groupby("dextran_channel", dropna=False)["p_um/s"]
                    .apply(lambda s: int(s.notna().sum()))
                    .to_dict()
                )
                self._append_log(f"[INFO] Non-null p_um/s rows by dextran_channel: {counts}")
        except Exception:
            pass

        self._results.append(batch_df)
        self._results_df = pd.concat(self._results, ignore_index=True)

        self._append_log(f"[OK] Saved all-image CSV to {output_csv}")
        display(batch_df.head(20))
        return batch_df

    def _save_results(self, output_csv: Path = Path("perfusion_results.csv")):
        if self._results_df is None or self._results_df.empty:
            self._append_log("[WARN] No results to save yet.")
            return None

        output_csv = self._normalize_csv_path(output_csv)
        output_csv.parent.mkdir(parents=True, exist_ok=True)
        self._results_df.to_csv(output_csv, index=False)
        self._append_log(f"[OK] Saved current results to {output_csv}")
        return output_csv


app = PerfusionNapariGUIApp()